In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

In [2]:
df = pd.read_csv("business.csv")

In [4]:
df.shape[0]

150346

In [9]:
# STEP 1 — select only businesses where categories contain the word "restaurant"
df['categories_clean'] = df['categories'].fillna("").str.lower()

df_rest = df[df['categories_clean'].str.contains("restaurant")]

print("Number of businesses that are restaurants:", df_rest.shape[0])

Number of businesses that are restaurants: 52286


In [11]:
cuisines = [
    "chinese", "italian", "japanese", "korean", "thai", "vietnamese",
    "indian", "greek", "mexican", "french", "american",
    "mediterranean", 
    "spanish", "fast food"
]

In [13]:
def find_cuisine(text):
    for c in cuisines:
        if c in text:
            return c
    return None

df_rest["cuisine"] = df_rest['categories_clean'].apply(find_cuisine)

# Filter out rows where no cuisine was found (optional)
df_final = df_rest[df_rest["cuisine"].notna()]

print("Number of restaurants with identifiable cuisine:", df_final.shape[0])
df_final[['business_id', 'name', 'cuisine', 'categories']].head()

Number of restaurants with identifiable cuisine: 33596


/var/folders/hr/6gkffdjd0313jpll9dlqqc1r0000gn/T/ipykernel_36551/780030968.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_rest["cuisine"] = df_rest['categories_clean'].apply(find_cuisine)


,business_id,name,cuisine,categories
5,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,fast food,"Burgers, Fast Food, Sandwiches, Food, Ice Crea..."
8,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,italian,"Pubs, Restaurants, Italian, Bars, American (Tr..."
9,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,fast food,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,..."
11,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,vietnamese,"Vietnamese, Food, Restaurants, Food Trucks"
12,il_Ro8jwPlHresjw9EGmBg,Denny's,american,"American (Traditional), Restaurants, Diners, B..."


In [17]:
# Count how many restaurants per cuisine
cuisine_counts = df_final['cuisine'].value_counts()

print(cuisine_counts)

cuisine
american         11495
italian           4553
mexican           4503
fast food         4124
chinese           3170
japanese          1486
indian             794
greek              704
thai               702
vietnamese         616
mediterranean      560
french             428
korean             347
spanish            114
Name: count, dtype: int64


In [19]:
df_final.shape

(33596, 16)

In [21]:
df_final.to_csv("business_cleaned.csv", index=False)


In [ ]:
## create dim_category table

In [31]:
df = df_final.copy()

# Split categories into lists
df['category_list'] = df['cuisine'].apply(lambda x: [c.strip() for c in x.split(",")])

# Explode so each row has ONE category
df_exploded = df.explode('category_list')

# Clean
df_exploded['category_list'] = df_exploded['category_list'].str.strip()

In [33]:
unique_categories = df_exploded['category_list'].dropna().unique()

# Start category keys at 10000
start_key = 10000
dim_category = pd.DataFrame({
    "category_key": range(start_key, start_key + len(unique_categories)),
    "category_name": unique_categories
})

In [35]:
dim_category.head(10)

,category_key,category_name
0,10000,fast food
1,10001,italian
2,10002,vietnamese
3,10003,american
4,10004,japanese
5,10005,korean
6,10006,chinese
7,10007,mexican
8,10008,french
9,10009,indian


In [37]:
dim_category.to_csv("dim_category.csv", index=False)


In [39]:
## create bridge_restaurant_category table

In [ ]:
df = pd.read_csv("business_cleaned.csv")

In [41]:
bridge = df_exploded.merge(dim_category,
                           left_on='category_list',
                           right_on='category_name')[['business_id', 'category_key']]

bridge = bridge.drop_duplicates()


In [43]:
bridge.head(10)

,business_id,category_key
0,CF33F8-E6oudUQ46HnavjQ,10000
1,k0hlBqXX-Bt0vf1op7Jr1w,10001
2,bBDDEgkFA1Otx9Lfe7BZUQ,10000
3,eEOYSgkmpB90uNA7lDOMRA,10002
4,il_Ro8jwPlHresjw9EGmBg,10003
5,0bPLkL0QhhPO5kt1_EXmNQ,10001
6,MUTTqe8uqyMdBl186RmNeA,10004
7,ROeacJQwBeh05Rqg7F6TCg,10005
8,9OG5YkX1g2GReZM0AskizA,10001
9,tMkwHmWFUEXrC9ZduonpTg,10004


In [45]:
bridge.to_csv("bridge_restaurant_category.csv", index=False)

In [ ]:
## create dim_user table

In [73]:
df_user = pd.read_csv("user_sample.csv")

In [53]:
df_user.head()

,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013","LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA...",52,...,13,10,17,3,66,96,119,119,35,18
3,SZDeASXq7o05mMNLshsdIA,Gwen,224,2005-11-29 04:38:33,512,330,299,"2009,2010,2011","enx1vVPnfdNUdPho6PH_wg, 4wOcvMLtU6a9Lslggq74Vg...",28,...,4,1,6,2,12,16,26,26,10,9
4,hA5lMy-EnncsH4JoR-hFGQ,Karen,79,2007-01-05 19:40:59,29,15,7,NaN,"PBK4q9KEEBHhFvSXCUirIw, 3FWPpM7KU1gXeOM_ZbYMbA...",1,...,1,0,0,0,1,1,0,0,0,0


In [75]:
dim_user = df_user[[
    "user_id",
    "name",
    "review_count",
    "average_stars",
    "fans",
    "useful",
    "funny",
    "cool",
    "yelping_since",
    "elite"
]].copy()

In [77]:
# convert yelping_since to DATE (drop time part)
dim_user["yelping_since"] = pd.to_datetime(dim_user["yelping_since"]).dt.date

print(dim_user.head())

                  user_id    name  review_count  average_stars  fans  useful  \
0  qVc8ODYU5SZjKXVBgXdI7w  Walker           585           3.91   267    7217   
1  j14WgRoU_-2ZE1aw1dXrJg  Daniel          4333           3.74  3138   43091   
2  2WnXYQFK0hXEoTxPtV2zvg   Steph           665           3.32    52    2086   
3  SZDeASXq7o05mMNLshsdIA    Gwen           224           4.27    28     512   
4  hA5lMy-EnncsH4JoR-hFGQ   Karen            79           3.54     1      29   

   funny   cool yelping_since  \
0   1259   5994    2007-01-25   
1  13066  27281    2009-01-25   
2   1010   1003    2008-07-25   
3    330    299    2005-11-29   
4     15      7    2007-01-05   

                                               elite  
0                                               2007  
1  2009,2010,2011,2012,2013,2014,2015,2016,2017,2...  
2                           2009,2010,2011,2012,2013  
3                                     2009,2010,2011  
4                                            

In [79]:
def count_elite_years(x):
    if pd.isna(x) or x == "":
        return 0
    return x.count(",") + 1

dim_user["elite_duration"] = dim_user["elite"].apply(count_elite_years)

In [83]:
dim_user = dim_user.drop(columns=["elite"])


In [85]:
dim_user.to_csv("dim_user.csv", index=False)

In [ ]:
## create dim_date table

In [93]:
df_review = pd.read_csv("review_sample.csv")
df_review["review_date"] = pd.to_datetime(df_review["date"]).dt.date


In [95]:
# 2. Create a DataFrame of unique dates
unique_dates = pd.DataFrame({"date": df_review["review_date"].unique()})

# 3. Convert to datetime type (again) for attribute extraction
unique_dates["date"] = pd.to_datetime(unique_dates["date"])


In [99]:
# Unique dates
unique_dates = pd.DataFrame({
    "full_date": pd.to_datetime(df_review["review_date"].unique())
})

# Build dim_date structure
dim_date = unique_dates.copy()

# (1) date_key = YYYYMMDD (integer)
dim_date["date_key"] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)

# (2) year, month, day
dim_date["year"] = dim_date["full_date"].dt.year
dim_date["month"] = dim_date["full_date"].dt.month
dim_date["day"] = dim_date["full_date"].dt.day

# (3) weekday (English or Chinese optional)
# English:
dim_date["weekday"] = dim_date["full_date"].dt.strftime("%A")

# (4) is_weekend (BOOLEAN)
dim_date["is_weekend"] = dim_date["weekday"].isin(["Saturday", "Sunday"]).astype(int)

# (5) week_number
dim_date["week_number"] = dim_date["full_date"].dt.isocalendar().week.astype(int)

# (6) quarter
dim_date["quarter"] = dim_date["full_date"].dt.quarter


dim_date = dim_date[[
    "date_key",
    "full_date",
    "year",
    "month",
    "day",
    "weekday",
    "is_weekend",
    "week_number",
    "quarter"
]]


dim_date.head()

,date_key,full_date,year,month,day,weekday,is_weekend,week_number,quarter
0,20180707,2018-07-07,2018,7,7,Saturday,1,27,3
1,20120103,2012-01-03,2012,1,3,Tuesday,0,1,1
2,20140205,2014-02-05,2014,2,5,Wednesday,0,6,1
3,20150104,2015-01-04,2015,1,4,Sunday,1,1,1
4,20170114,2017-01-14,2017,1,14,Saturday,1,2,1


In [101]:
# Save CSV
dim_date.to_csv("dim_date.csv", index=False)

In [103]:
## create Fact table

In [113]:
# Load review data
df = pd.read_csv("review_sample.csv")

In [115]:
df.head()

,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15


In [117]:
# (1) Convert date column to datetime
df["review_date"] = pd.to_datetime(df["date"])

# (2) Create date_key (YYYYMMDD)
df["date_key"] = df["review_date"].dt.strftime("%Y%m%d").astype(int)

# (3) Compute text_length
df["text_length"] = df["text"].astype(str).apply(len)

# (4) Build final fact_review dataframe
fact_review = df[[
    "review_id",
    "business_id",
    "user_id",
    "date_key",
    "stars",
    "useful",
    "funny",
    "cool",
    "text_length"
]]

fact_review.head()

,review_id,business_id,user_id,date_key,stars,useful,funny,cool,text_length
0,KU_O5udG6zpxOg-VcAEodg,XQfwVwDr-v0ZS3_CbbE5Xw,mh_-eMZ6K5RLWhZyISBhwA,20180707,3,0,0,0,513
1,BiTunyQ73aT9WBnpR9DZGw,7ATYjTIgM3jUlt4UM3IypQ,OyoGAe7OKpv6SyGZT5g77Q,20120103,5,1,0,1,829
2,saUsX_uimxRlCVr67Z4Jig,YjUWPpI6HXG530lwP-fb2A,8g_iMtfSiwikVnbP2etR0A,20140205,3,0,0,0,339
3,AqPFMleE6RsU23_auESxiA,kxX2SOes4o-D3ZQBkiMRfA,_7bHUi9Uuf5__HHc_Q8guQ,20150104,5,1,0,1,243
4,Sx8TMOWLNuJBWer-0pcmoA,e4Vwtrqf-wpJfwesgvdgxQ,bcjbaE6dDog4jkNY91ncLQ,20170114,4,1,0,1,534


In [119]:
fact_review.to_csv("fact_review.csv", index=False)

In [159]:
# Load restaurant data
df_rest = pd.read_csv("dim_restaurant.csv")

# Load location/key mapping
df_loc = pd.read_csv("dim_location.csv")

In [161]:
# Merge restaurant data with location mapping
df_merged = df_rest.merge(
    df_loc[['city_name', 'city_key']], 
    left_on='city', 
    right_on='city_name',
    how='left'
)

In [163]:
# Drop the redundant city_name column
df_merged = df_merged.drop(columns=['city_name'])

# Drop old city column and rename city_key
df_merged = df_merged.drop(columns=['city'])

In [167]:
df_merged['city_key'] = df_merged['city_key'].astype('Int64')


In [169]:
df_merged.head()

,business_id,name,address,state,postal_code,latitude,longitude,stars,review_count,is_open,category,city_key
0,PdMXmOWDRHICAx6SLgu1dQ,24,2401 Walnut St,PA,19103.0,39.951521,-75.179873,3.5,111,0,italian,1478
1,jHtqYimdAVzlZEtmmoLY7w,211,211 N Tampa St,FL,33602.0,27.945968,-82.457590,3.5,32,1,american,1620
2,wu6SSj2rw9Cc9frUmexEng,943,943 S 9th St,PA,19147.0,39.938189,-75.157992,3.5,80,0,italian,1478
3,H9YalVFugskHF38MfJWT_w,@Ramen,1608 Sansom St,PA,19103.0,39.950476,-75.167718,2.5,12,0,japanese,1478
4,Ix_88gFalSxSCsz5Fbnl7A,@Ramen,4357 Main St,PA,19127.0,40.025762,-75.223853,4.0,190,1,japanese,1478


In [209]:
df_merged.to_csv("dim_restaurant.csv", index=False, encoding="utf-8")

In [215]:
import unicodedata

def normalize_ascii(text):
    if pd.isna(text):
        return text
    
    # 1. Replace smart quotes with normal apostrophe
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    
    # 2. Normalize accents (é → e, À → A)
    text = unicodedata.normalize('NFKD', text)
    
    # 3. Remove remaining non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    return text

# Apply to name + address (other columns optional)
df["name"] = df["name"].apply(normalize_ascii)

In [217]:
df.head(10)

,business_id,name,address,state,postal_code,latitude,longitude,stars,review_count,is_open,category,city_key
0,PdMXmOWDRHICAx6SLgu1dQ,24,2401 Walnut St,PA,19103.0,39.951521,-75.179873,3.5,111,0,italian,1478.0
1,jHtqYimdAVzlZEtmmoLY7w,211,211 N Tampa St,FL,33602.0,27.945968,-82.457590,3.5,32,1,american,1620.0
2,wu6SSj2rw9Cc9frUmexEng,943,943 S 9th St,PA,19147.0,39.938189,-75.157992,3.5,80,0,italian,1478.0
3,H9YalVFugskHF38MfJWT_w,@Ramen,1608 Sansom St,PA,19103.0,39.950476,-75.167718,2.5,12,0,japanese,1478.0
4,Ix_88gFalSxSCsz5Fbnl7A,@Ramen,4357 Main St,PA,19127.0,40.025762,-75.223853,4.0,190,1,japanese,1478.0
5,oCVcvXmtVJKAH8vpFCoVyg,#1 Mongolian BBQ - Best Stir Fried Noodles In ...,"8249 W Overland Rd, Ste 180",ID,83709.0,43.589722,-116.285309,3.5,51,1,chinese,1053.0
6,tzAVcGlQrCXSLdj7lZq3zg,CUATRO,5300 W Lutz Lake Fern Rd,FL,33558.0,28.154217,-82.529269,3.5,26,1,american,1348.0
7,34UYYxLV_lAxHYNnV3GXAg,ALAVITA,807 W Idaho St,ID,83702.0,43.616384,-116.203200,4.0,305,1,italian,1053.0
8,TKpt60GHbIZEt8aA3AB9Cg,Ardana Food & Drink,751 Easton Rd,PA,18976.0,40.232501,-75.136615,4.5,82,1,mediterranean,1679.0
9,y9-HGTuihVxjkI6Bh_2k7Q,026 Bar and Grill,515 Gravois Rd,MO,63026.0,38.513450,-90.437967,3.5,7,0,american,1193.0


In [223]:
def contains_non_ascii(text):
    try:
        text.encode("ascii")
        return False   # good row → ASCII
    except UnicodeEncodeError:
        return True    # bad row → contains unicode

non_ascii_rows = df[df["name"].apply(contains_non_ascii)]



non_ascii_rows


,business_id,name,address,state,postal_code,latitude,longitude,stars,review_count,is_open,category,city_key


In [225]:
df.to_csv("dim_restaurant.csv", index=False, encoding="utf-8")

In [ ]:
## filter fact review

In [227]:
# Load dimension + fact data
dim_rest = pd.read_csv("dim_restaurant.csv")      
fact = pd.read_csv("fact_review.csv")   

In [229]:
valid_business_ids = set(dim_rest["business_id"])

fact_clean = fact[fact["business_id"].isin(valid_business_ids)].copy()

print("Original fact_review rows:", len(fact))
print("After filtering by business_id:", len(fact_clean))

Original fact_review rows: 200000
After filtering by business_id: 98512


In [233]:
len(dim_rest)

31738

In [231]:
fact_clean.to_csv("fact_review.csv", index=False, encoding="utf-8")


In [237]:
# 1. Load both files
fact = pd.read_csv("fact_review.csv")          # or fact_review.csv if that's what you're using
rest = pd.read_csv("dim_restaurant.csv")       # or dim_restaurant_cleaned.csv

# 2. Clean business_id in BOTH (strip spaces, tabs, etc.)
fact["business_id"] = fact["business_id"].astype(str).str.strip()
rest["business_id"] = rest["business_id"].astype(str).str.strip()

# 3. Compare sets
fact_ids = set(fact["business_id"])
rest_ids = set(rest["business_id"])

missing_in_rest = fact_ids - rest_ids   # business_ids used in fact but not present in dim_restaurant

print("Total unique business_id in fact_review:", len(fact_ids))
print("Total unique business_id in dim_restaurant:", len(rest_ids))
print("Number of business_id in fact with NO match in dim_restaurant:", len(missing_in_rest))

# 4. Show some problematic IDs
print("Sample mismatched business_ids:", list(missing_in_rest)[:20])

# 5. See some rows from fact that are causing trouble
problem_rows = fact[fact["business_id"].isin(missing_in_rest)]
print("Sample problematic rows from fact_review:")
print(problem_rows.head(10))

Total unique business_id in fact_review: 2728
Total unique business_id in dim_restaurant: 31738
Number of business_id in fact with NO match in dim_restaurant: 0
Sample mismatched business_ids: []
Sample problematic rows from fact_review:
Empty DataFrame
Columns: [review_id, business_id, user_id, date_key, stars, useful, funny, cool, text_length]
Index: []


In [239]:
rest = pd.read_csv("dim_restaurant.csv")  # or your final restaurant CSV
print(len(rest))
print(rest["business_id"].nunique())

31738
31738


In [243]:
print("Total rows:", len(rest))
print("Rows with city_key NULL:", rest["city_key"].isna().sum())

Total rows: 31738
Rows with city_key NULL: 82
